# Temporal diffusion analysis with SHE

This notebook loads realistic interaction data from a JSON-Lines file,
builds a hyperstructure, slices it into temporal windows, and shows
how diffusion patterns and bridge structures evolve over time.

**Scenario:** Six accounts across two communities discuss two topics
(energy and housing) over three time periods.  A bridge triad gradually
forms between the communities.

In [ ]:
from pathlib import Path
import logging
logging.getLogger('she.diffusion').setLevel(logging.ERROR)

from she import (
    SHEHyperstructure,
    SHEConfig,
    rank_diffusers,
    rank_entity_diffusers,
    find_bridge_simplices,
    group_cohesion,
    rank_influencers,
    rolling_windows,
)

## 1. Load data and build hyperstructure

In [ ]:
data_path = Path('..') / 'data' / 'interactions.jsonl'

config = SHEConfig(max_dimension=2, spectral_k=5)
hs = SHEHyperstructure.from_jsonl(data_path, name='social_temporal', config=config)

# Tag communities manually (in practice this would come from profile data)
for name in ['anna', 'ben', 'carol']:
    hs.add_entity(name, community='energy_cluster')
for name in ['dave', 'elena', 'frank']:
    hs.add_entity(name, community='housing_cluster')

print(hs)

## 2. Global analysis

Before looking at time slices, check the overall structure.

In [ ]:
print('--- Top entity diffusers ---')
for r in rank_entity_diffusers(hs, top_k=6):
    label = r.target[0]
    print(f'  {label:>8s}  score={r.score:.3f}  {r.metadata}')

print('\n--- Bridge simplices ---')
for b in find_bridge_simplices(hs, top_k=5):
    print(f'  {sorted(b.members)}  communities={b.communities_spanned}  '
          f'bridge={b.bridge_score:.3f}  kind={b.metadata.get("kind","")}')

print('\n--- Graph vs simplex ---')
comp = rank_influencers(hs, top_k=3)
print('Graph centrality:', [(r.target[0], round(r.score, 3)) for r in comp['graph_centrality']])
print('Simplex diffusion top-3:')
for r in comp['simplex_diffusion'][:3]:
    print(f'  dim={r.dimension}  {r.target}  score={r.score:.3f}')

## 3. Temporal windowing

Slice the interaction log into three periods and watch the bridge form.

In [ ]:
snapshots = rolling_windows(hs, window_size=1.0, step=1.0, bounds=(1.0, 4.0))

for start, end, ws in snapshots:
    print(f'\n=== Period t=[{start:.0f}, {end:.0f}) ===')
    print(f'  Entities: {sorted(ws.entities)}')
    print(f'  Relations: {len(ws.relations)}')
    
    bridges = find_bridge_simplices(ws)
    if bridges:
        print(f'  Bridges:')
        for b in bridges[:3]:
            print(f'    {sorted(b.members)}  communities={b.communities_spanned}  '
                  f'score={b.bridge_score:.3f}')
    else:
        print('  Bridges: none')
    
    top = rank_entity_diffusers(ws, top_k=3)
    if top:
        print(f'  Top diffusers: {[(r.target[0], round(r.score, 3)) for r in top]}')

## 4. Group cohesion over time

Track how the cross-community triad {carol, dave, elena} gains cohesion.

In [ ]:
bridge_group = ['carol', 'dave', 'elena']

print(f'Group cohesion for {bridge_group}:\n')
for start, end, ws in snapshots:
    # only score if all members present
    if all(m in ws.entities for m in bridge_group):
        cs = group_cohesion(ws, bridge_group)
        print(f'  t=[{start:.0f},{end:.0f})  score={cs.score:.4f}  {cs.components}')
    else:
        print(f'  t=[{start:.0f},{end:.0f})  group not fully present')

## 5. Takeaway

In period 1 the two communities are almost separate, connected only by a
weak carol-dave mention.  By period 3, {carol, dave, elena} has become a
cohesive cross-community triad that bridges the energy and housing clusters.

Graph centrality never highlights this triad -- it ranks individuals.
SHE's bridge detection and group cohesion scoring make the formation of
this higher-order bridge structure visible across time.